# 03 — The complete Transformer from first principles

> **Status:** in progress; embeddings and sinusoidal positions are complete.

- **Mapped issue:** [#5](https://github.com/majorgilles/transformer-2017-reproduction/issues/5)
- **Depends on:** `02_shared_bpe.ipynb` / issue #3.
- **Paper reference:** sections 3.1–3.5, *Attention Is All You Need*.


In [1]:
#| default_exp model


## Goal

Build the complete encoder-decoder Transformer in prerequisite order, from token IDs and positions through explicit attention, six-layer stacks, logits, and a tiny overfit.


## Token IDs become vectors

A token ID selects one learned row from an embedding table. The table has one row
per vocabulary token and `d_model` columns per row.

In [2]:
import torch
from torch import nn

torch.manual_seed(0)

token_ids = torch.tensor([[0, 2, 1]])
embedding = nn.Embedding(num_embeddings=4, embedding_dim=6)
token_vectors = embedding(token_ids)

print(f"token ID shape: {tuple(token_ids.shape)}")
print(f"embedding table shape: {tuple(embedding.weight.shape)}")
print(f"token vector shape: {tuple(token_vectors.shape)}")
print(token_vectors)

token ID shape: (1, 3)
embedding table shape: (4, 6)
token vector shape: (1, 3, 6)
tensor([[[-1.1258, -1.1524, -0.2506, -0.4339,  0.8487,  0.6920],
         [ 0.1665,  0.8744, -0.1435, -0.1116,  0.9318,  1.2590],
         [-0.3160, -2.1152,  0.4681, -0.1577,  1.4437,  0.2660]]],
       grad_fn=<EmbeddingBackward0>)


## Why scale token embeddings?

The Transformer combines token identity and position by adding their vectors
element by element. Sinusoidal position values are fixed between approximately
`-1` and `1`, while token embeddings are learned.

Following section 3.4 of *Attention Is All You Need*, we multiply each token
embedding by:

$$
\sqrt{d_{\text{model}}}
$$

This width-aware scale gives the token signal a useful magnitude before the
position vector is added. It does not change the token, vector shape, or
direction; it only changes the vector's magnitude.

In [3]:
#| export
import torch
from torch import nn


class TokenEmbedding(nn.Module):
    """Look up token vectors and apply the paper's embedding scale.

    Each token ID selects one learned vector with `d_model` values. Following
    section 3.4 of the Transformer paper, the vector is multiplied by
    `sqrt(d_model)` before positional information is added.
    """

    def __init__(self, vocab_size: int, d_model: int) -> None:
        super().__init__()
        # One learned row per vocabulary token; each row has `d_model` values.
        self.embedding = nn.Embedding(vocab_size, d_model)
        # Paper section 3.4: strengthen token identity relative to the fixed
        # sinusoidal values that will be added to this vector.
        self.scale = d_model**0.5

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Return scaled vectors with shape (batch, sequence, d_model)."""
        # Scaling changes magnitude but preserves shape and token order.
        return self.embedding(token_ids) * self.scale

In [4]:
torch.manual_seed(0)

scaled_embedding = TokenEmbedding(vocab_size=4, d_model=6)
raw_vectors = scaled_embedding.embedding(token_ids)
scaled_vectors = scaled_embedding(token_ids)

expected_vectors = raw_vectors * scaled_embedding.scale

assert scaled_vectors.shape == (1, 3, 6)
assert torch.allclose(scaled_vectors, expected_vectors)

raw_norm = raw_vectors[0, 0].norm().item()
scaled_norm = scaled_vectors[0, 0].norm().item()

print("Embedding scaling demonstration:")
print(f"  scale: {scaled_embedding.scale:.3f}")
print(f"  shape before/after scaling: {tuple(scaled_vectors.shape)}")
print(f"  first-token norm before: {raw_norm:.3f}")
print(f"  first-token norm after: {scaled_norm:.3f}")

Embedding scaling demonstration:
  scale: 2.449
  shape before/after scaling: (1, 3, 6)
  first-token norm before: 2.011
  first-token norm after: 4.927


## Embeddings alone do not represent order

An embedding lookup depends only on the token ID. If the same token appears at
three positions, all three positions receive the same vector. The Transformer
therefore needs a separate position signal before attention can distinguish
where each occurrence appears.

In [5]:
repeated_token_ids = torch.tensor([[2, 2, 2]])
repeated_vectors = scaled_embedding(repeated_token_ids)

assert torch.allclose(repeated_vectors[0, 0], repeated_vectors[0, 1])
assert torch.allclose(repeated_vectors[0, 1], repeated_vectors[0, 2])

print("Repeated-token evidence:")
print(f"  token IDs: {repeated_token_ids.tolist()}")
print(f"  first three values at position 0: {repeated_vectors[0, 0, :3]}")
print(f"  first three values at position 1: {repeated_vectors[0, 1, :3]}")
print(f"  first three values at position 2: {repeated_vectors[0, 2, :3]}")

Repeated-token evidence:
  token IDs: [[2, 2, 2]]
  first three values at position 0: tensor([ 0.4077,  2.1418, -0.3514], grad_fn=<SliceBackward0>)
  first three values at position 1: tensor([ 0.4077,  2.1418, -0.3514], grad_fn=<SliceBackward0>)
  first three values at position 2: tensor([ 0.4077,  2.1418, -0.3514], grad_fn=<SliceBackward0>)


## Sinusoidal positions create an order signal

For position `pos` and dimension pair `i`, the paper defines:

$$
PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

$$
PE(pos, 2i + 1) = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

Each sine/cosine pair changes at a different speed. Early dimensions change
quickly across nearby positions, while later dimensions change slowly. Together,
these patterns give every position a distinct vector with `d_model` values.

### Reading the table slices

A slice uses `start:stop:step`. When `stop` is omitted, Python continues to
the end of that axis. In `position_table[:, 0::2]`, the first `:` selects every
position row, while `0::2` selects even columns `0, 2, 4, ...`. Those columns
receive sine values. Similarly, `position_table[:, 1::2]` selects odd columns
`1, 3, 5, ...`, which receive cosine values.

For `d_model = 6`, the columns form three sine/cosine pairs: `(0, 1)`, `(2, 3)`,
and `(4, 5)`.

In [6]:
max_length = 4
d_model = 6

# `pos` in the paper: positions 0, 1, 2, 3.
positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
print(positions.shape)

print(d_model // 2)
# `i` in the paper: one index per sine/cosine pair.
pair_indices = torch.arange(d_model // 2, dtype=torch.float32)

# The exact denominator from 10000^(2i / d_model).
denominators = 10000.0 ** ((2 * pair_indices) / d_model)

# The angle inside both sin(...) and cos(...).
angles = positions / denominators

position_table = torch.zeros(max_length, d_model)

# PE(pos, 2i): keep every position row (`:`) and fill even columns
# 0, 2, 4, ... (`0::2`) with sine values.
position_table[:, 0::2] = torch.sin(angles)

# PE(pos, 2i + 1): keep every row and fill odd columns
# 1, 3, 5, ... (`1::2`) with cosine values.
position_table[:, 1::2] = torch.cos(angles)

assert position_table.shape == (4, 6)
assert torch.allclose(
    position_table[0],
    torch.tensor([0.0, 1.0, 0.0, 1.0, 0.0, 1.0]),
)

print("Sinusoidal position table:")
print(position_table)

torch.Size([4, 1])
3
Sinusoidal position table:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000]])


In [7]:
#| export
def sinusoidal_position_table(max_length: int, d_model: int) -> torch.Tensor:
    """Build the paper's fixed sine/cosine position vectors.

    The returned tensor has shape `(max_length, d_model)`. Each row represents
    one sequence position. Even columns contain sine values and their adjacent
    odd columns contain cosine values computed with the same frequency.
    """

    if d_model % 2 != 0:
        raise ValueError("d_model must be even so sine/cosine dimensions can pair")

    positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
    pair_indices = torch.arange(d_model // 2, dtype=torch.float32)

    # This directly represents the paper's denominator: 10000^(2i / d_model).
    denominators = 10000.0 ** ((2 * pair_indices) / d_model)
    angles = positions / denominators

    table = torch.zeros(max_length, d_model)

    # PE(pos, 2i): `:` keeps every position row; `0::2` selects even
    # dimensions 0, 2, 4, ... for sine values.
    table[:, 0::2] = torch.sin(angles)

    # PE(pos, 2i + 1): `1::2` selects the adjacent odd dimensions
    # 1, 3, 5, ... for cosine values at the same frequencies.
    table[:, 1::2] = torch.cos(angles)

    return table

In [8]:
reusable_position_table = sinusoidal_position_table(
    max_length=4,
    d_model=6,
)

assert torch.allclose(reusable_position_table, position_table)

print("Reusable position table matches the paper-formula experiment:")
print(reusable_position_table)

Reusable position table matches the paper-formula experiment:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000]])


## Add fixed positions to token vectors

The position table is fixed rather than learned. We register it as a PyTorch
buffer so it moves with the model and is saved with the model, but receives no
optimizer updates.

For an input with shape `(batch, sequence, d_model)`, we select the first
`sequence` rows from the table and add a batch axis. The resulting shape
`(1, sequence, d_model)` broadcasts across every item in the batch.

In [9]:
#| export
class SinusoidalPositionalEncoding(nn.Module):
    """Add the paper's fixed sinusoidal position vectors to token vectors.

    The fixed table is stored as a buffer rather than a trainable parameter.
    During `forward`, the module selects one table row per token position and
    broadcasts those rows across every sequence in the batch.
    """

    position_table: torch.Tensor

    def __init__(self, max_length: int, d_model: int) -> None:
        super().__init__()

        table = sinusoidal_position_table(
            max_length=max_length,
            d_model=d_model,
        )

        # A buffer is saved and moved with the module but is not learned.
        self.register_buffer("position_table", table)

    def forward(self, token_vectors: torch.Tensor) -> torch.Tensor:
        """Add positions to a `(batch, sequence, d_model)` tensor."""

        sequence_length = token_vectors.shape[1]

        if sequence_length > self.position_table.shape[0]:
            raise ValueError("sequence length exceeds the position table")

        # Select one position row per token, then add a batch axis so the same
        # position pattern broadcasts across every sequence in the batch.
        positions = self.position_table[:sequence_length].unsqueeze(0)

        return token_vectors + positions

### Proving what the module adds

The module computes `combined = tokens + positions`. Subtracting the original
token vectors therefore recovers the position vectors: `combined - tokens =
positions`. Comparing those recovered offsets with the position table proves
that the module selects the correct rows, broadcasts them across the batch, and
does not otherwise alter the token vectors.

Because these calculations use 32-bit floating-point numbers, addition followed
by subtraction can leave tiny rounding differences. `torch.allclose` uses an
absolute tolerance of `1e-6` here: differences smaller than one millionth are
accepted, while a wrong position or formula still fails.

In [14]:
positional_encoding = SinusoidalPositionalEncoding(
    max_length=4,
    d_model=6,
)

# Repeat the same three-token sequence to create a batch of two.
token_batch = scaled_vectors.repeat(2, 1, 1)
print(token_batch.shape)
combined_vectors = positional_encoding(token_batch)

# Subtraction reveals exactly which position vectors were added.
position_offsets = combined_vectors - token_batch

# Select the three expected table rows, add a batch axis, and explicitly
# expand them to the same `(2, 3, 6)` shape as the recovered offsets.
expected_offsets = reusable_position_table[:3].unsqueeze(0).expand_as(position_offsets)

assert combined_vectors.shape == (2, 3, 6)

# Floating-point addition and subtraction can leave differences smaller than
# one millionth. The tolerance ignores that noise, not real formula errors.
assert torch.allclose(
    position_offsets,
    expected_offsets,
    rtol=0.0,
    atol=1e-6,
)

# The table belongs to the module as a fixed buffer, with no learned parameters.
assert "position_table" in dict(positional_encoding.named_buffers())
assert list(positional_encoding.parameters()) == []

print("Token + position demonstration:")
print(f"  token batch shape: {tuple(token_batch.shape)}")
print(f"  combined shape: {tuple(combined_vectors.shape)}")
print("  first three offset values at each position:")
for position in range(3):
    print(f"    position {position}: {position_offsets[0, position, :3]}")

torch.Size([2, 3, 6])
Token + position demonstration:
  token batch shape: (2, 3, 6)
  combined shape: (2, 3, 6)
  first three offset values at each position:
    position 0: tensor([0., 1., 0.], grad_fn=<SliceBackward0>)
    position 1: tensor([0.8415, 0.5403, 0.0464], grad_fn=<SliceBackward0>)
    position 2: tensor([ 0.9093, -0.4161,  0.0927], grad_fn=<SliceBackward0>)


## What this notebook demonstrates

- Token IDs select learned vectors with `d_model` values.
- Token embeddings are multiplied by $\sqrt{d_{\text{model}}}$.
- Fixed sine/cosine vectors give each sequence position an order signal.
- Token and position vectors combine without changing tensor shape.


## Focused evidence

- Embedding output has shape `(batch, sequence, d_model)`.
- Scaling changes vector magnitude but preserves shape and token order.
- Repeated token IDs receive identical vectors before positions are added.
- Position zero alternates between `sin(0) = 0` and `cos(0) = 1`.
- The positional table is a fixed buffer with no learned parameters.
- Subtracting token vectors from combined vectors recovers the expected positions.


## Scaled dot-product attention

Attention lets each query position gather information from the available value
positions. A query is what one token is looking for, a key describes a possible
match, and a value contains the information returned from that match.

The paper defines:

$$
\operatorname{Attention}(Q, K, V)
=
\operatorname{softmax}
\left(
\frac{QK^\mathsf{T}}{\sqrt{d_k}}
\right)V
$$

### Tensor contract

All inputs except `mask` are floating-point tensors.

- `query`: `(..., query_length, d_k)`
- `key`: `(..., key_length, d_k)`
- `value`: `(..., key_length, d_v)`
- `mask`: Boolean tensor broadcastable to `(..., query_length, key_length)`

The leading `...` dimensions may later contain the batch and attention-head
dimensions.

The function returns:

- `output`: `(..., query_length, d_v)`
- `probabilities`: `(..., query_length, key_length)`

Each probability row describes how one query distributes its attention across
all key positions. A `True` mask entry blocks that query-key relationship.


In [ ]:
#| export
def scaled_dot_product_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute explicit scaled dot-product attention.

    Args:
        query: Floating-point tensor shaped `(..., query_length, d_k)`.
        key: Floating-point tensor shaped `(..., key_length, d_k)`.
        value: Floating-point tensor shaped `(..., key_length, d_v)`.
        mask: Optional Boolean tensor broadcastable to
            `(..., query_length, key_length)`. `True` entries are blocked.

    Returns:
        A pair containing:
        - contextual vectors shaped `(..., query_length, d_v)`;
        - attention probabilities shaped
          `(..., query_length, key_length)`.
    """
    d_k = key.shape[-1]

    scores = query @ key.transpose(-2, -1)
    scores = scores / d_k**0.5

    if mask is not None:
        scores = scores.masked_fill(mask, -torch.inf)

    probabilities = torch.softmax(scores, dim=-1)
    output = probabilities @ value

    return output, probabilities

### Direct self-attention smoke check

Before adding learned head projections, run the attention function directly on
the token-plus-position vectors produced above. The dimensions are deliberately
small so the full probability matrix remains visible; the function itself accepts
arbitrary batch sizes, sequence lengths, and vector widths.

Tensor types and shapes in this check:

- `combined_vectors`: floating-point `(batch=2, num_tokens=3, d_model=6)`
- queries, keys, and values: the same floating-point tensor for self-attention
- `self_attention_probabilities`: floating-point `(2, 3, 3)`
- `self_attention_output`: floating-point `(2, 3, 6)`

Each `(3, 3)` probability matrix contains one row per querying token and one
column per key token. Every row must sum to one.


In [13]:
print(combined_vectors.shape)
self_attention_output, self_attention_probabilities = scaled_dot_product_attention(
    query=combined_vectors,
    key=combined_vectors,
    value=combined_vectors,
)

assert self_attention_output.shape == (2, 3, 6)
assert self_attention_probabilities.shape == (2, 3, 3)
assert torch.allclose(
    self_attention_probabilities.sum(dim=-1),
    torch.ones(2, 3),
)
assert torch.isfinite(self_attention_output).all()

print("Self-attention probabilities for the first sequence:")
print(self_attention_probabilities[0])

torch.Size([2, 3, 6])
Self-attention probabilities for the first sequence:
tensor([[6.9697e-01, 1.3346e-03, 3.0170e-01],
        [6.2607e-05, 9.9993e-01, 2.9187e-06],
        [1.2384e-05, 2.5539e-09, 9.9999e-01]], grad_fn=<SelectBackward0>)


## Multi-head attention

Multi-head attention learns several views of the same model vectors. Four learned
linear projections create queries, keys, values, and the final output. Each of the
first three projections keeps the total width `d_model`, then reshapes that width
into `num_heads` independent parts of width `d_head = d_model // num_heads`.

Inputs are floating-point tensors, and the optional mask is Boolean:

- `query`: `(batch, query_length, d_model)`
- `key`: `(batch, key_length, d_model)`
- `value`: `(batch, key_length, d_model)`
- `mask`: broadcastable to `(batch, num_heads, query_length, key_length)`

The internal tensor flow is:

| Stage | Shape |
| --- | --- |
| projected queries | `(batch, query_length, d_model)` |
| split queries | `(batch, num_heads, query_length, d_head)` |
| split keys/values | `(batch, num_heads, key_length, d_head)` |
| attention probabilities | `(batch, num_heads, query_length, key_length)` |
| attended values | `(batch, num_heads, query_length, d_head)` |
| joined heads | `(batch, query_length, d_model)` |
| projected output | `(batch, query_length, d_model)` |

Self-attention passes one sequence as all three inputs. Encoder-decoder
cross-attention passes decoder vectors as queries and encoder vectors as keys and
values, so `query_length` and `key_length` may differ.


In [ ]:
#| export
class MultiHeadAttention(nn.Module):
    """Paper-style multi-head self-attention and cross-attention.

    Inputs use `(batch, sequence, d_model)`. Internally, projections use
    `(batch, num_heads, sequence, d_head)`, where
    `d_head = d_model // num_heads`.
    """

    d_model: int
    num_heads: int
    d_head: int
    query_projection: nn.Linear
    key_projection: nn.Linear
    value_projection: nn.Linear
    output_projection: nn.Linear

    def __init__(self, d_model: int, num_heads: int) -> None:
        super().__init__()

        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.query_projection = nn.Linear(d_model, d_model)
        self.key_projection = nn.Linear(d_model, d_model)
        self.value_projection = nn.Linear(d_model, d_model)
        self.output_projection = nn.Linear(d_model, d_model)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Run attention over batched model vectors.

        Args:
            query: `(batch, query_length, d_model)`.
            key: `(batch, key_length, d_model)`.
            value: `(batch, key_length, d_model)`.
            mask: Boolean tensor broadcastable to
                `(batch, num_heads, query_length, key_length)`.

        Returns:
            Output vectors shaped `(batch, query_length, d_model)` and
                probabilities shaped
               `(batch, num_heads, query_length, key_length)`.
        """
        # Read the dimensions needed to reshape the projected tensors.
        # query: (batch, query_length, d_model)
        # key/value: (batch, key_length, d_model)
        batch_size = query.shape[0]
        query_length = query.shape[1]
        key_length = key.shape[1]

        # Learn separate query, key, and value representations. Each linear
        # projection preserves its input shape and total width `d_model`.
        # projected_queries: (batch, query_length, d_model)
        # projected_keys/values: (batch, key_length, d_model)
        projected_queries = self.query_projection(query)
        projected_keys = self.key_projection(key)
        projected_values = self.value_projection(value)

        # Split `d_model` into `num_heads * d_head`, then move the head axis
        # before the sequence axis so every head attends independently.
        # queries: (batch, num_heads, query_length, d_head)
        queries = projected_queries.reshape(
            batch_size,
            query_length,
            self.num_heads,
            self.d_head,
        ).transpose(1, 2)

        # keys: (batch, num_heads, key_length, d_head)
        keys = projected_keys.reshape(
            batch_size,
            key_length,
            self.num_heads,
            self.d_head,
        ).transpose(1, 2)

        # values: (batch, num_heads, key_length, d_head)
        values = projected_values.reshape(
            batch_size,
            key_length,
            self.num_heads,
            self.d_head,
        ).transpose(1, 2)

        # Every head compares each query with every key and retrieves a
        # weighted mixture of that head's value vectors.
        # attended_values: (batch, num_heads, query_length, d_head)
        # probabilities: (batch, num_heads, query_length, key_length)
        attended_values, probabilities = scaled_dot_product_attention(
            query=queries,
            key=keys,
            value=values,
            mask=mask,
        )

        # Move heads beside one another and concatenate their `d_head` values
        # back into one `d_model` vector for each query position.
        # joined_heads: (batch, query_length, d_model)
        joined_heads = attended_values.transpose(1, 2).reshape(
            batch_size,
            query_length,
            self.d_model,
        )

        # Let the model learn how to combine information from all heads.
        # output: (batch, query_length, d_model)
        output = self.output_projection(joined_heads)

        return output, probabilities

## Position-wise feed-forward network

After attention exchanges information between token positions, the feed-forward
network transforms each token vector independently. The same two learned linear
layers are reused at every batch item and sequence position.

The paper defines:

$$
\operatorname{FFN}(x)
=
\max(0, xW_1 + b_1)W_2 + b_2
$$

Tensor flow:

| Stage | Type and shape |
| --- | --- |
| input | floating point `(batch, sequence, d_model)` |
| expanded features | floating point `(batch, sequence, d_ff)` |
| ReLU activation | floating point `(batch, sequence, d_ff)` |
| output | floating point `(batch, sequence, d_model)` |

Unlike attention, this operation does not mix sequence positions.

In [ ]:
#| export
class PositionwiseFeedForward(nn.Module):
    """Apply the paper's two-layer ReLU network at every token position."""

    d_model: int
    d_ff: int
    input_projection: nn.Linear
    output_projection: nn.Linear

    def __init__(self, d_model: int, d_ff: int) -> None:
        super().__init__()

        self.d_model = d_model
        self.d_ff = d_ff

        self.input_projection = nn.Linear(d_model, d_ff)
        self.output_projection = nn.Linear(d_ff, d_model)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """Transform `(batch, sequence, d_model)` token vectors."""
        # Expand each token independently.
        # expanded: (batch, sequence, d_ff)
        expanded = self.input_projection(inputs)

        # Keep positive activations and replace negative values with zero.
        # activated: (batch, sequence, d_ff)
        activated = torch.relu(expanded)

        # Return each token to the model width.
        # output: (batch, sequence, d_model)
        output = self.output_projection(activated)

        return output

## Encoder layer: self-attention, residuals, and post-norm

One encoder layer contains two sublayers:

1. multi-head self-attention;
2. a position-wise feed-forward network.

The 2017 Transformer wraps each sublayer with dropout, a residual addition, and
then layer normalization:

$$
\operatorname{LayerNorm}
\left(
x + \operatorname{Dropout}(\operatorname{Sublayer}(x))
\right)
$$

This is **post-norm** because normalization happens after the sublayer output is
added to its input.

Tensor flow:

| Stage | Type and shape |
| --- | --- |
| encoder input | floating point `(batch, source_length, d_model)` |
| self-attention output | floating point `(batch, source_length, d_model)` |
| first residual/post-norm | floating point `(batch, source_length, d_model)` |
| feed-forward output | floating point `(batch, source_length, d_model)` |
| encoder-layer output | floating point `(batch, source_length, d_model)` |
| source mask | Boolean, broadcastable to `(batch, num_heads, source_length,
 source_length)` |

In [ ]:
#| export
class EncoderLayer(nn.Module):
    """One paper-faithful post-norm Transformer encoder layer."""

    self_attention: MultiHeadAttention
    feed_forward: PositionwiseFeedForward
    attention_dropout: nn.Dropout
    attention_norm: nn.LayerNorm
    feed_forward_dropout: nn.Dropout
    feed_forward_norm: nn.LayerNorm

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.self_attention = MultiHeadAttention(
            d_model=d_model,
            num_heads=num_heads,
        )
        self.feed_forward = PositionwiseFeedForward(
            d_model=d_model,
            d_ff=d_ff,
        )

        self.attention_dropout = nn.Dropout(dropout)
        self.attention_norm = nn.LayerNorm(d_model)

        self.feed_forward_dropout = nn.Dropout(dropout)
        self.feed_forward_norm = nn.LayerNorm(d_model)

    def forward(
        self,
        inputs: torch.Tensor,
        source_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """Encode `(batch, source_length, d_model)` vectors."""
        # Self-attention lets every source token gather information from the
        # source sequence.
        # attention_output: (batch, source_length, d_model)
        # probabilities:
        #   (batch, num_heads, source_length, source_length)
        attention_output, _ = self.self_attention(
            query=inputs,
            key=inputs,
            value=inputs,
            mask=source_mask,
        )

        # Paper-faithful post-norm:
        # LayerNorm(inputs + Dropout(attention_output))
        # after_attention: (batch, source_length, d_model)
        after_attention = self.attention_norm(inputs + self.attention_dropout(attention_output))

        # Transform each contextualized source token independently.
        # feed_forward_output: (batch, source_length, d_model)
        feed_forward_output = self.feed_forward(after_attention)

        # Second paper-faithful post-norm residual path.
        # output: (batch, source_length, d_model)
        output = self.feed_forward_norm(
            after_attention + self.feed_forward_dropout(feed_forward_output)
        )

        return output

## Six-layer encoder stack

The base Transformer passes the source representation through six encoder
layers. Every layer has the same architecture but owns independent attention,
feed-forward, and normalization parameters.

$$
\operatorname{Encoder}(x)
=
\operatorname{EncoderLayer}_6(
\ldots
\operatorname{EncoderLayer}_2(
\operatorname{EncoderLayer}_1(x)
))
$$

Tensor flow:

| Stage | Type and shape |
| --- | --- |
| encoder input | floating point `(batch, source_length, d_model)` |
| each layer output | floating point `(batch, source_length, d_model)` |
| encoder output | floating point `(batch, source_length, d_model)` |
| source mask | Boolean, broadcastable to `(batch, num_heads, source_length,
source_length)` |

The sequence length and model width remain unchanged across the stack.

In [ ]:
%%sql


In [ ]:
#| export
class Encoder(nn.Module):
    layers: nn.ModuleList

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float = 0.1,
        num_layers: int = 6,
    ) -> None:
        super().__init__()

        # Construct a new EncoderLayer on every iteration. Do not repeat one
        # existing layer object, because all layers must own separate weights.
        self.layers = nn.ModuleList(
            [
                EncoderLayer(
                    d_model=d_model,
                    num_heads=num_heads,
                    d_ff=d_ff,
                    dropout=dropout,
                )
                for _ in range(num_layers)
            ]
        )

    def forward(
        self, inputs: torch.Tensor, source_mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        """Encode `(batch, source_length, d_model)` source vectors."""
        # output: (batch, source_length, d_model)
        output = inputs

        # Every iteration preserves the tensor shape while applying a new set
        # of self-attention, FFN, and normalization parameters.
        for layer in self.layers:
            output = layer(
                inputs=output,
                source_mask=source_mask,
            )

        # output: (batch, source_length, d_model)
        return output

## Decoder attention masks

Decoder self-attention produces scores shaped:

```text
(batch, num_heads, target_length, target_length)
```

In [ ]:
#| export
def make_padding_mask(
    token_ids: torch.Tensor,
    pad_token_id: int,
) -> torch.Tensor:
    """Block padding keys in `(batch, sequence_length)` token IDs."""
    # token_ids == pad_token_id: (batch, sequence_length)
    padding_positions = token_ids.eq(pad_token_id)

    # Add head and query axes so this broadcasts against attention scores.
    # mask: (batch, 1, 1, sequence_length)
    mask = padding_positions[:, None, None, :]

    return mask


#| export
def make_causal_mask(
    sequence_length: int,
    device: torch.device | None = None,
) -> torch.Tensor:
    """Block future target keys for every target query."""
    # Start with one Boolean decision for every query-key pair.
    # future_positions: (sequence_length, sequence_length)
    future_positions = torch.triu(
        torch.ones(
            sequence_length,
            sequence_length,
            dtype=torch.bool,
            device=device,
        ),
        diagonal=1,
    )

    # Reuse this rule across all batches and attention heads.
    # mask: (1, 1, sequence_length, sequence_length)
    mask = future_positions[None, None, :, :]

    return mask

In [ ]:
#| export
class DecoderLayer(nn.Module):
    """One paper-faithful post-norm Transformer decoder layer."""

    masked_multihead_attention: MultiHeadAttention
    masked_multihead_attention_dropout: nn.Dropout
    masked_multihead_attention_norm: nn.LayerNorm
    cross_attention: MultiHeadAttention
    cross_attention_dropout: nn.Dropout
    cross_attention_norm: nn.LayerNorm
    feed_forward: PositionwiseFeedForward
    feed_forward_dropout: nn.Dropout
    feed_forward_norm: nn.LayerNorm

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.masked_multihead_attention = MultiHeadAttention(
            d_model=d_model,
            num_heads=num_heads,
        )
        self.masked_multihead_attention_dropout = nn.Dropout(dropout)
        self.masked_multihead_attention_norm = nn.LayerNorm(d_model)

        self.cross_attention = MultiHeadAttention(
            d_model=d_model,
            num_heads=num_heads,
        )
        self.cross_attention_dropout = nn.Dropout(dropout)
        self.cross_attention_norm = nn.LayerNorm(d_model)

        self.feed_forward = PositionwiseFeedForward(
            d_model=d_model,
            d_ff=d_ff,
        )
        self.feed_forward_dropout = nn.Dropout(dropout)
        self.feed_forward_norm = nn.LayerNorm(d_model)

    def forward(
        self,
        inputs: torch.Tensor,
        memory: torch.Tensor,
        target_mask: torch.Tensor | None = None,
        source_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """Decode target vectors using encoded source memory."""
        # Masked self-attention lets each target token gather information only
        # from target positions permitted by the padding and causal mask.
        # masked_multihead_attention_output: (batch, target_length, d_model)
        masked_multihead_attention_output, _ = self.masked_multihead_attention(
            query=inputs,
            key=inputs,
            value=inputs,
            mask=target_mask,
        )

        # Preserve the original target state through the first residual path,
        # then apply paper-faithful post-norm.
        # after_multihead_attention: (batch, target_length, d_model)
        after_multihead_attention = self.masked_multihead_attention_norm(
            inputs + self.masked_multihead_attention_dropout(masked_multihead_attention_output)
        )

        # Cross-attention uses decoder states as queries and encoder memory as
        # keys and values, letting each target position read the source.
        # cross_attention_output: (batch, target_length, d_model)
        cross_attention_output, _ = self.cross_attention(
            query=after_multihead_attention,
            key=memory,
            value=memory,
            mask=source_mask,
        )

        # Preserve the post-self-attention decoder state through the second
        # residual path, then apply post-norm.
        # after_cross_attention: (batch, target_length, d_model)
        after_cross_attention = self.cross_attention_norm(
            after_multihead_attention + self.cross_attention_dropout(cross_attention_output)
        )

        # Transform each source-aware target position independently without
        # mixing sequence positions.
        # feed_forward_output: (batch, target_length, d_model)
        feed_forward_output = self.feed_forward(after_cross_attention)

        # Preserve the source-aware decoder state through the third residual
        # path, then apply the final post-norm.
        # output: (batch, target_length, d_model)
        output = self.feed_forward_norm(
            after_cross_attention + self.feed_forward_dropout(feed_forward_output)
        )

        return output

## Remaining model sections

The following sections complete this authoritative model notebook:

1. padding and causal masks inside explicit scaled dot-product attention;
2. multi-head self-attention and encoder-decoder cross-attention;
3. the position-wise ReLU FFN and paper-faithful post-norm residual path;
4. six-layer encoder and decoder stacks; and
5. tied full-model logits, a deterministic tiny overfit, and greedy output.

## Explicitly deferred

Label smoothing, the Noam optimizer schedule, campaign training, checkpoint resume, beam search, final evaluation, and publication.


## Review checkpoint

Review each mechanism in context, then review the integrated tensor flow, paper-fidelity choices, focused assertions, and tiny-overfit result before closing issue #5.
